# Biblical Qwen3.5 4B Fine-Tuning with Unsloth (4-bit QLoRA)

**Base Model:** `unsloth/Qwen3.5-4B` — bf16 weights (~8.7 GB), `Qwen3_5ForConditionalGeneration`,
`model_type` `qwen3_5`. Quantized to bnb NF4 on the fly by `load_in_4bit=True`. Already present in the
shared HF cache on this machine.

**Architecture (verified from the checkpoint's own `config.json` and `model.safetensors.index.json`, not assumed):**

| | |
|---|---|
| Language layers | 32 = **24 gated-delta `linear_attn`** + **8 `full_attention`** (`full_attention_interval: 4`) |
| Hidden size / heads | 2560 / 16 attn heads, 4 KV heads, head_dim 256 |
| Vision tower | `model.visual.*`, 24 blocks, 297 tensors — leaf names `attn.qkv`, `mlp.linear_fc1/2` |
| MTP head | `mtp.*`, 1 layer, 15 tensors on disk — leaf names **collide** with the language model's (`q_proj` … `down_proj`). Verified on this machine: `transformers 5.16.0.dev0` does **not** materialize it in this load path (0 `mtp` parameters), so the collision is latent, not active. |
| Vocab | 248,320 |
| `eos_token` (tokenizer) | `<\|im_end\|>` = 248046 |
| `config.text_config.eos_token_id` | **248044 = `<\|endoftext\|>` — disagrees with the chat template.** See the GGUF cells. |

This is the same *multimodal hybrid* class as Qwen3.8-27B, just small. Three consequences drive the
cells below: LoRA must be scoped away from the MTP head, packing must be off, and GGUF export must
override EOS.

**Dataset:** `biblical_personas_combined_sharegpt.jsonl` — 4,352 multi-turn ShareGPT conversations, each
carrying its own persona system prompt. Measured through this tokenizer with `enable_thinking=False`:
**p50 1575, p90 2138, p99 2494, max 2999 tokens.** `MAX_SEQ_LENGTH = 3072` truncates nothing.

**Training Hardware:** NVIDIA DGX Spark (GB10, sm_120, 128 GB unified memory).

**Chat Template:** Tokenizer-native Qwen ChatML (`<\|im_start\|>role\ncontent<\|im_end\|>`), applied via
`tokenizer.apply_chat_template`.

**Thinking mode — this template differs from Qwen3.8's.** Verified by rendering it: `enable_thinking`
here gates *only* the generation prompt, and it defaults to **off**. There is no "Reasoning effort is
set to ..." instruction to leak into the system prompt, unlike Qwen3.8. The formatting cell still
passes `enable_thinking=False` explicitly — omitting a kwarg and passing `False` happening to agree
today is not a property worth relying on — and the formatting cell asserts the system prompts came
through byte-identical rather than trusting that claim.

**Deployment:** The PEFT LoRA adapter is the shipping artifact. The GGUF cells at the end are optional.

**Lineage:** Recipe carried from
`biblical/notebooks/loras/qwen_38_27b/biblical_qwen3_8_27b_sft_unsloth_4bit.ipynb` (confirmed 544/544
steps on this machine against the same architecture family) with the durability fixes from
`stoic/notebooks/loras/qwen3.8/stoic_qwen38_27b_sft.ipynb` (water-filled stratified cap, completion
sentinel, GGUF EOS override) folded in. Its DPO counterpart is
`biblical_qwen3_5_4b_dpo_unsloth_4bit.ipynb` in this directory.

## How to run

Run cells in order. Cell 2 (Environment Preparation) is idempotent — if it installs anything,
**restart the kernel and resume from Cell 1**. Training auto-resumes from the newest
`checkpoint-*` in the train directory, so an interrupted run continues rather than restarting.

## 1. Configuration

In [1]:
import os

# =========================== ALLOCATOR: DELIBERATELY UNSET ===========================
# Do NOT set PYTORCH_CUDA_ALLOC_CONF here, and in particular do not set max_split_size_mb.
#
# docs/dgx_spark_gb10_quirks.md recommends
#     "garbage_collection_threshold:0.5,max_split_size_mb:256"
# and states the confirmed-working Biblical Qwen3.8 27B run sets it. That statement is wrong:
# neither biblical_qwen3_8_27b_sft_unsloth_4bit.ipynb nor stoic_qwen38_27b_sft.ipynb contains
# PYTORCH_CUDA_ALLOC_CONF anywhere, and it is not set in the container environment either. The
# runs that actually completed on this machine set nothing.
#
# Setting it here caused a reproducible OOM. max_split_size_mb:256 tells the caching allocator
# never to split a block larger than 256 MB. With a 248,320-token vocabulary the logits tensor is
# multiple GB, and because packing=False every batch pads to its own longest member, that tensor
# is a DIFFERENT SIZE almost every step. So a cached 5.8 GB block can never be carved down to
# serve a 5.3 GB request - the allocator must cudaMalloc a fresh one while the old one stays
# reserved. Each new batch shape permanently adds another multi-GB block.
#
# The signature is unmistakable once you look for it: peak ALLOCATION stayed near 20 GB while
# RESERVED climbed 24.9 -> 38.1 GB in eight steps. Live memory was never the problem; the cache
# was. On GB10 host and device share one 128 GB pool, so that climb consumes the machine.
#
# Also deliberately NO torch.cuda.set_per_process_memory_fraction(): on unified memory a
# fractional cap protects nothing and adds a ceiling the bf16 GGUF merge would have to fit under.
# If fragmentation ever does need addressing, "expandable_segments:True" is the setting designed
# for varying allocation sizes - max_split_size_mb is the opposite of what this workload wants.

# =========================== PATHS (all cascade from PROJECT_ROOT) ===========================
# Normally run inside the `unsloth-notebook` container, which bind-mounts
# /home/spark/projects/training -> /workspace/training. The fallbacks let the same notebook run
# on the host without edits.
if os.path.exists("/workspace/training/biblical"):
    PROJECT_ROOT = "/workspace/training/biblical"
elif os.path.exists("/workspace/biblical"):
    PROJECT_ROOT = "/workspace/biblical"
else:
    PROJECT_ROOT = "/home/spark/projects/training/biblical"

OUTPUT_ROOT = f"{PROJECT_ROOT}/output"

# =========================== SHARED HUGGING FACE CACHE ===========================
# Point the Hub cache at the cache shared with vLLM so the base is downloaded once and reused.
# unsloth/Qwen3.5-4B is already present there (~8.7 GB). Must be set BEFORE unsloth/transformers
# are imported.
for _cache_dir in ("/root/.cache/huggingface", "/home/spark/.cache/huggingface"):
    if os.path.isdir(_cache_dir):
        os.environ["HF_HUB_CACHE"] = _cache_dir
        break

# =========================== MODEL CONFIGURATION ===========================
# unsloth/Qwen3.5-4B ships bf16 weights (Qwen3_5ForConditionalGeneration, model_type qwen3_5).
# There is no pre-quantized bnb-4bit repo for it, so Unsloth quantizes to NF4 on the fly via
# load_in_4bit=True in the model cell.
BASE_LLM = "unsloth/Qwen3.5-4B"

# NOTE: biblical_qwen3_5_4b_dpo_unsloth_4bit.ipynb reads this exact string as SFT_MODEL_NAME_BASE
# to locate the adapter it continues from. Change it in both places or not at all.
MODEL_NAME_BASE = "biblical_qwen3_5_4b_unsloth_4bit"

# =========================== THINKING MODE ===========================
# Verified against this checkpoint's chat_template.jinja, which differs from Qwen3.8's:
#
#   {%- if add_generation_prompt %}
#       {{- '<|im_start|>assistant\n' }}
#       {%- if enable_thinking is defined and enable_thinking is true %}
#           {{- '<think>\n' }}          <- opens an unclosed reasoning block
#       {%- else %}
#           {{- '<think>\n\n</think>\n\n' }}   <- empty block; answer directly
#       {%- endif %}
#   {%- endif %}
#
# Two things follow, both confirmed by rendering the template:
#   1. `enable_thinking` affects ONLY add_generation_prompt=True. Training text is identical
#      with the kwarg omitted or set to False.
#   2. This template injects NO reasoning-effort instruction into the system prompt. Qwen3.8's
#      does; this one does not. The formatting cell asserts that rather than assuming it.
#
# This flag controls only the inference/eval cells. Training formatting is hardcoded to False so
# the two can never silently diverge.
ENABLE_THINKING = False

# =========================== INPUT DATA ===========================
# Combined multi-turn ShareGPT JSONL from the datagen notebooks (per-persona + augmented).
# Already quality-filtered, multi-turn, grouped by topic. 4,352 conversations.
INPUT_DATA_FILE = f"{PROJECT_ROOT}/data/training-data/biblical_persona_v2/biblical_personas_combined_sharegpt.jsonl"

# =========================== PERSONA SYSTEM PROMPTS ===========================
# System prompts are EXTRACTED from the JSONL at load time (see the data loading cell). This keeps
# training in sync with datagen — regenerate data with new/changed prompts and this notebook picks
# them up automatically. After loading, `persona_system_prompts` maps persona_key -> full prompt
# text, and is saved alongside the LoRA adapters for inference.

# =========================== OUTPUT DIRECTORIES ===========================
OUTPUT_BASE_DIR = f"{OUTPUT_ROOT}/{MODEL_NAME_BASE}"
OUTPUT_DIR_ADAPTERS = f"{OUTPUT_BASE_DIR}/train"
LORA_OUTPUT_DIR = f"{OUTPUT_BASE_DIR}/lora_adapters"
FORMATTED_DATASET_DIR = f"{OUTPUT_DIR_ADAPTERS}/formatted_sft_dataset_cache"

# =========================== TRAINING HYPERPARAMETERS ===========================
# MAX_SEQ_LENGTH measured, not guessed: all 4,352 conversations rendered through THIS tokenizer
# with enable_thinking=False give p50=1575, p90=2138, p99=2494, max=2999 tokens. 3072 truncates
# nothing. Packing is off and batches pad to their own longest member, so the headroom is free.
MAX_SEQ_LENGTH = 3072

# DO NOT RAISE BATCH_SIZE. It is 2 because of the VOCABULARY, not the weights.
#
# The 4B's weights are only ~2.5 GB in NF4, which invites the reasoning "a 4B has room for a
# bigger batch than the 27B needed." That reasoning is wrong here and it was measured to be wrong
# on this machine: at BATCH_SIZE=4 the run reached 19.7 GB peak allocation and 38.1 GB reserved
# after 8 steps, still climbing, on its way to consuming the whole 128 GB pool.
#
# The cause is that Qwen3.5's vocabulary is 248,320 tokens, so the logits tensor is
# batch x seq x 248320 and dwarfs everything else:
#     batch 2, seq 3072 -> 6.1 GB fp32   (+ a bf16 copy + a same-shaped gradient)
#     batch 4, seq 3072 -> 12.2 GB fp32  (+ copy + gradient)
# Nothing chunks this. And because packing=False every batch has a different shape, so the
# caching allocator accumulates blocks it cannot reuse and `reserved` ratchets upward for the
# whole run rather than settling.
#
# 2 x 4 is the per-device batch and effective batch of the confirmed-working Qwen3.8 27B run,
# which completed 544/544 steps at this same MAX_SEQ_LENGTH and the same 248K vocab.
BATCH_SIZE = 2
GRAD_ACCUM = 4

LEARNING_RATE = 2e-4

# 1, matching both confirmed-working 27B SFT notebooks. A second epoch is defensible for a model
# this small, but every gratuitous deviation from the recipe that actually runs on this machine
# has cost a failed run so far. Raise it once a full pass has completed cleanly.
TARGET_EPOCHS = 1

SFT_MAX_EXAMPLES = 0  # 0 = use all valid conversations; set e.g. 500/1000/3000 for capped runs

# Set True to rebuild the formatted-dataset cache even when its manifest matches.
FORCE_REFORMAT = False

# =========================== CHECKPOINTING ===========================
# 4,352 examples x 2 epochs / effective batch 8 = ~1,088 steps. Checkpoint often enough that a
# crash costs minutes, not the whole run; the training cell auto-resumes from the newest.
SAVE_STEPS = 50
SAVE_TOTAL_LIMIT = 3

# gc.collect() + torch.cuda.empty_cache() cadence during training. Aligned with the checkpoint
# interval — right after a save is when transient buffers are safe to drop. Periodic, NOT
# per-step: empty_cache() forces the allocator to re-acquire blocks, and doing that every step
# measurably slows training while buying nothing.
CLEANUP_STEPS = 50

# =========================== LoRA CONFIGURATION ===========================
LORA_R = 32
LORA_ALPHA = 32
LORA_DROPOUT = 0

# These leaf names are the standard attention + MLP projections. On Qwen3.5-4B, scoped to the
# language model, they resolve to:
#     96 MLP modules   (gate/up/down x all 32 layers)
#   + 32 self_attn     (q/k/v/o x the 8 full_attention layers only)
#   = 128 modules
# The 24 gated-delta `linear_attn` layers use different leaf names (in_proj_qkv / in_proj_a /
# in_proj_b / in_proj_z / out_proj) and are deliberately NOT adapted — their MLPs are. This
# mirrors the confirmed-working Qwen3.8 adapter; adding the linear-attention projections would be
# a separate, deliberate experiment.
#
# IMPORTANT: the LoRA cell pairs this list with finetune_vision_layers=False. Without that flag
# Unsloth hands the bare list to PEFT, which matches by SUFFIX across the whole module tree. See
# the LoRA cell's markdown for exactly what that does and does not reach on this checkpoint.
LORA_TARGET_MODULES = [
    "q_proj", "k_proj", "v_proj", "o_proj",
    "gate_proj", "up_proj", "down_proj",
]

# =========================== INFERENCE TEST ===========================
# This repo publishes no generation_config.json for Qwen3.5-4B, so these are the project's
# standard Qwen sampling values used across the other Biblical notebooks — not values read from
# the checkpoint.
TEST_PROMPT = "I am struggling with forgiveness. What does Scripture teach about forgiving others?"
GEN_TEMPERATURE = 0.7
GEN_TOP_P = 0.8
GEN_TOP_K = 20

# ============================================================================
print("Configuration loaded (Qwen3.5 4B 4-bit QLoRA, biblical persona SFT)")
print(f"  Project root:     {PROJECT_ROOT}")
print(f"  HF hub cache:     {os.environ.get('HF_HUB_CACHE', '<default>')}")
print(f"  Base model:       {BASE_LLM}")
print(f"  Model name:       {MODEL_NAME_BASE}")
print(f"  Input data:       {INPUT_DATA_FILE}")
print(f"  Output base:      {OUTPUT_BASE_DIR}")
print(f"  LoRA output:      {LORA_OUTPUT_DIR}")
print(f"  Formatted cache:  {FORMATTED_DATASET_DIR}")
print(f"  LoRA config:      r={LORA_R}, alpha={LORA_ALPHA}, targets={LORA_TARGET_MODULES}")
print(f"  Training:         batch={BATCH_SIZE}, grad_accum={GRAD_ACCUM} "
      f"(effective {BATCH_SIZE * GRAD_ACCUM}), lr={LEARNING_RATE}, epochs={TARGET_EPOCHS}")
print(f"  SFT max examples: {SFT_MAX_EXAMPLES or 'ALL'}")
print(f"  Max seq length:   {MAX_SEQ_LENGTH} (measured dataset max is 2999 tokens)")
print(f"  Checkpoints:      every {SAVE_STEPS} steps, keep {SAVE_TOTAL_LIMIT}")
print(f"  Thinking mode:    {'ON' if ENABLE_THINKING else 'OFF'} (inference/eval cells only)")
print(f"  Persona prompts:  extracted from JSONL at load time")

if not os.path.exists(INPUT_DATA_FILE):
    raise FileNotFoundError(f"Training data not found: {INPUT_DATA_FILE}")

Configuration loaded (Qwen3.5 4B 4-bit QLoRA, biblical persona SFT)
  Project root:     /workspace/training/biblical
  HF hub cache:     /root/.cache/huggingface
  Base model:       unsloth/Qwen3.5-4B
  Model name:       biblical_qwen3_5_4b_unsloth_4bit
  Input data:       /workspace/training/biblical/data/training-data/biblical_persona_v2/biblical_personas_combined_sharegpt.jsonl
  Output base:      /workspace/training/biblical/output/biblical_qwen3_5_4b_unsloth_4bit
  LoRA output:      /workspace/training/biblical/output/biblical_qwen3_5_4b_unsloth_4bit/lora_adapters
  Formatted cache:  /workspace/training/biblical/output/biblical_qwen3_5_4b_unsloth_4bit/train/formatted_sft_dataset_cache
  LoRA config:      r=32, alpha=32, targets=['q_proj', 'k_proj', 'v_proj', 'o_proj', 'gate_proj', 'up_proj', 'down_proj']
  Training:         batch=2, grad_accum=4 (effective 8), lr=0.0002, epochs=1
  SFT max examples: ALL
  Max seq length:   3072 (measured dataset max is 2999 tokens)
  Checkpoints: 

## 2. Environment Preparation

Idempotent — safe to re-run. Order matters:

1. Verify the CUDA PyTorch build is intact (no side effects).
2. Core training packages, and remove the aarch64 `torchao` build that blocks PEFT's bnb 4-bit dispatcher.
3. `transformers` + `peft` from git main — **only if the installed `transformers` does not already
   register the `qwen3_5` architecture.** The container currently ships `transformers 5.16.0.dev0`,
   which does, so this normally skips. Reinstalling a working build from git main is a real way to
   break a working container.
4. Small utility packages.
5. Rebuild `causal_conv1d` from source if its compiled CUDA extension is missing — the NGC image
   ships the Python package **without** it, which hard-crashes any import reaching the FalconH1
   model inside `transformers`/`unsloth`. If the build fails, uninstall it so the PyTorch fallback
   path is taken cleanly.
6. `flash-linear-attention` — Triton kernels for the gated-delta-rule fast path used by this
   model's 24 `linear_attn` layers.
7. Import `unsloth` **before** `transformers` so its monkey-patches apply.

**If this cell installs anything, restart the kernel and resume from Cell 1.**

In [2]:
import os, sys, subprocess, importlib, importlib.util

_installed_something = False


def _pip(*args, env_extra=None):
    """Run pip against this kernel's interpreter; print output only on failure."""
    global _installed_something
    env = os.environ.copy()
    if env_extra:
        env.update(env_extra)
    result = subprocess.run(
        [sys.executable, "-m", "pip", *args], capture_output=True, text=True, env=env
    )
    if result.returncode != 0:
        print(f"  PIP FAILED: {' '.join(args)}")
        print(result.stderr[-500:] if result.stderr else result.stdout[-500:])
        return False
    _installed_something = True
    return True


def _check_import(module_name):
    try:
        return importlib.import_module(module_name)
    except (ImportError, ModuleNotFoundError):
        return None


print("=" * 60)
print("ENVIRONMENT SETUP")
print("=" * 60)

# --- 1. Verify the CUDA PyTorch build is intact ------------------------------
import torch
if not torch.cuda.is_available():
    print("FATAL: torch.cuda.is_available() = False")
    print(f"  torch version: {torch.__version__}")
    if "cpu" in torch.__version__:
        print("  CUDA PyTorch was clobbered by pip. Recreate the container.")
    else:
        print("  GPU not passed through. Check runtime=nvidia, NVIDIA_VISIBLE_DEVICES=all")
    raise RuntimeError("No GPU. Cannot continue. See messages above.")
print(f"  torch {torch.__version__} - CUDA {torch.version.cuda} - GPU: {torch.cuda.get_device_name(0)}")

# --- 2. Core training packages ------------------------------------------------
print("  Ensuring core packages (unsloth, trl, accelerate, datasets, bitsandbytes)...")
_pip("install", "-q", "-U", "unsloth", "trl", "accelerate", "datasets", "bitsandbytes")

# Container ships torchao 0.14.0+git (custom aarch64 build). peft requires torchao>=0.16.0 OR
# torchao absent. No aarch64 wheel >=0.16 exists on PyPI, so uninstall - peft's torchao dispatcher
# then no-ops and falls through to the bnb 4-bit dispatcher, which is what QLoRA wants anyway.
if importlib.util.find_spec("torchao") is not None:
    print("  Removing torchao (blocks peft's bnb 4-bit dispatcher on aarch64)...")
    _pip("uninstall", "-y", "-q", "torchao")

# mergekit 0.1.4 cannot be imported under this container's pydantic 2.13. Its merge methods are
# declared with pydantic.create_model(), and 2.13 now builds a schema for the generated `execute`
# callable, whose Dict[str, torch.Tensor] argument raises
#     PydanticSchemaGenerationError: Unable to generate pydantic-core schema for torch.Tensor
# That import is not optional for us: unsloth's patch_trl_vision_model_mapping imports
# trl.trainer.dpo_trainer -> trl.trainer.callbacks -> trl.mergekit_utils -> mergekit, so
# `import unsloth` itself dies in Step 7. TRL only needs mergekit for MergeModelCallback, which
# this notebook never uses. Reinstall with `pip install mergekit==0.1.4` if anything else wants it.
if importlib.util.find_spec("mergekit") is not None:
    print("  Removing mergekit (pydantic 2.13 breaks its schema build, killing `import unsloth`)...")
    _pip("uninstall", "-y", "-q", "mergekit")

# --- 3. transformers + peft from git main, ONLY if qwen3_5 is unknown ---------
# `qwen3_5` is the architecture behind Qwen3.5 / 3.6 / 3.8. If the installed transformers already
# registers it, there is nothing to gain from a git-main reinstall and a real chance of breaking a
# working build, so this is conditional rather than unconditional.
try:
    from transformers.models.auto.configuration_auto import CONFIG_MAPPING_NAMES
    _qwen35_known = "qwen3_5" in CONFIG_MAPPING_NAMES
except Exception:
    _qwen35_known = False

if _qwen35_known:
    print("  transformers already registers `qwen3_5` - skipping git-main reinstall")
else:
    print("  `qwen3_5` not registered - installing transformers + peft from git main...")
    _pip("install", "-q", "-U", "git+https://github.com/huggingface/transformers.git")
    _pip("install", "-q", "-U", "git+https://github.com/huggingface/peft.git")

# --- 4. Small utility packages ------------------------------------------------
for _module, _install_args in {
    "psutil":      ["install", "-q", "psutil"],
    "matplotlib":  ["install", "-q", "matplotlib"],
    "ipywidgets":  ["install", "-q", "ipywidgets"],
    "torchvision": ["install", "-q", "--no-deps", "torchvision"],
    "PIL":         ["install", "-q", "pillow"],
}.items():
    if _check_import(_module) is None:
        print(f"  Installing {_install_args[-1]}...")
        _pip(*_install_args)

# --- 5. Fix causal_conv1d -----------------------------------------------------
# The NGC image installs the causal_conv1d Python package WITHOUT its compiled CUDA extension
# (causal_conv1d_cuda). That hard-crashes any import that reaches the FalconH1 model inside
# transformers or unsloth - i.e. `import unsloth` itself - so it must be fixed BEFORE either.
#
# pip also caches a broken prebuilt aarch64 wheel, so --no-binary is required to force a source
# build, together with CAUSAL_CONV1D_FORCE_BUILD=TRUE. The first build takes a few minutes on
# aarch64; pip caches the result afterwards.
_causal_ok = False
_build_env = {
    "CAUSAL_CONV1D_FORCE_BUILD": "TRUE",
    "TORCH_CUDA_ARCH_LIST": "12.0;12.1",   # DGX Spark GB10 = sm_120
}
try:
    from causal_conv1d.causal_conv1d_interface import causal_conv1d_fn
    _causal_ok = True
    print("  causal_conv1d: OK (CUDA extension loaded)")
    # Clean up - don't leave partial imports that spoil unsloth's import order
    for _k in list(sys.modules.keys()):
        if "causal_conv1d" in _k:
            del sys.modules[_k]
except (ImportError, ModuleNotFoundError, OSError):
    print("  causal_conv1d: CUDA extension missing - rebuilding from source (~3 min)...")
    _pip("uninstall", "-y", "causal-conv1d")
    _pip("cache", "remove", "causal_conv1d")
    for _k in list(sys.modules.keys()):
        if "causal_conv1d" in _k:
            del sys.modules[_k]
    importlib.invalidate_caches()
    _ok = _pip("install", "--no-build-isolation", "--no-deps", "--force-reinstall",
               "--no-binary", "causal-conv1d", "causal-conv1d", env_extra=_build_env)
    if _ok:
        importlib.invalidate_caches()
        try:
            from causal_conv1d.causal_conv1d_interface import causal_conv1d_fn
            _causal_ok = True
            print("  causal_conv1d: rebuilt OK (CUDA extension working)")
            for _k in list(sys.modules.keys()):
                if "causal_conv1d" in _k:
                    del sys.modules[_k]
        except (ImportError, ModuleNotFoundError, OSError):
            print("  causal_conv1d: rebuild produced no CUDA ext - uninstalling for fallback")
            _pip("uninstall", "-y", "causal-conv1d")
            for _k in list(sys.modules.keys()):
                if "causal_conv1d" in _k:
                    del sys.modules[_k]
            importlib.invalidate_caches()
    else:
        print("  causal_conv1d: source build failed - uninstalling for fallback")
        _pip("uninstall", "-y", "causal-conv1d")
        importlib.invalidate_caches()

# --- 6. flash-linear-attention ------------------------------------------------
# fla provides the Triton JIT kernels (chunk_gated_delta_rule etc.) used by this model's 24
# gated-delta linear_attn layers. fla-core installs into the same `fla` namespace.
if _check_import("fla") is None:
    print("  Installing flash-linear-attention...")
    _pip("install", "-q", "--no-deps", "flash-linear-attention", "fla-core")

_fast_path_ok = False
try:
    from fla.ops.gated_delta_rule import chunk_gated_delta_rule, fused_recurrent_gated_delta_rule
    _fast_path_ok = _causal_ok and chunk_gated_delta_rule is not None
    for _k in list(sys.modules.keys()):
        if _k.startswith("fla."):
            del sys.modules[_k]
except (ImportError, ModuleNotFoundError):
    pass
print(f"  Fast path: {'ENABLED' if _fast_path_ok else 'DISABLED (using torch fallback)'}")

# --- 7. Import unsloth FIRST, then transformers -------------------------------
# Unsloth must be imported before transformers/trl/peft so its monkey-patches apply. Purge
# anything already loaded so the imports pick up whatever was installed above.
for _k in list(sys.modules.keys()):
    if _k in ("transformers", "trl", "peft") or _k.startswith(("transformers.", "trl.", "peft.")):
        del sys.modules[_k]
importlib.invalidate_caches()

import unsloth
import transformers
import peft
import trl

print()
print(f"  unsloth:       {unsloth.__version__}")
print(f"  transformers:  {transformers.__version__}")
print(f"  peft:          {peft.__version__}")
print(f"  trl:           {trl.__version__}")
print(f"  causal_conv1d: {'OK' if _causal_ok else 'FALLBACK (torch path)'}")
print(f"  fla:           {'OK' if _check_import('fla') else 'MISSING'}")
print(f"  torchao:       {'PRESENT (should be absent)' if importlib.util.find_spec('torchao') else 'absent (correct)'}")
print(f"  mergekit:      {'PRESENT (should be absent)' if importlib.util.find_spec('mergekit') else 'absent (correct)'}")
print()
if _installed_something:
    print("Packages were installed/removed. RESTART THE KERNEL, then rerun from Cell 1.")
else:
    print("Environment already satisfied - no installs, no restart needed. Continue to Cell 3.")

ENVIRONMENT SETUP
  torch 2.10.0a0+b558c986e8.nv25.11 - CUDA 13.0 - GPU: NVIDIA GB10
  Ensuring core packages (unsloth, trl, accelerate, datasets, bitsandbytes)...
  Removing torchao (blocks peft's bnb 4-bit dispatcher on aarch64)...
  transformers already registers `qwen3_5` - skipping git-main reinstall
  causal_conv1d: OK (CUDA extension loaded)
  Fast path: ENABLED
🦥 Unsloth: Will patch your computer to enable 2x faster free finetuning.
🦥 Unsloth Zoo will now patch everything to make training faster!

  unsloth:       2026.8.22
  transformers:  5.5.0
  peft:          0.20.1.dev0
  trl:           0.24.0
  causal_conv1d: OK
  fla:           OK
  torchao:       absent (correct)

Packages were installed/removed. RESTART THE KERNEL, then rerun from Cell 1.


## 3. Load Dataset

Load the combined multi-turn ShareGPT JSONL from the datagen notebooks.

- Already quality-filtered (no short answers, no AI refusals)
- Multi-turn: QA pairs grouped per conversation by topic
- Each conversation carries a persona-specific system prompt
- Standard ShareGPT format: `[system, human, gpt, human, gpt, ...]`

In [3]:
import json, os, re, random
from collections import defaultdict
from datasets import Dataset as HFDataset

print(f"LOADING COMBINED SHAREGPT DATA")
print(f"  File: {INPUT_DATA_FILE}")

# Load multi-turn conversations and EXTRACT system prompts from the JSONL. This replaces
# hardcoded prompt dicts — prompts stay in sync with datagen automatically.
conversations = []
persona_system_prompts = {}   # persona_key -> full system prompt text
persona_counts = defaultdict(int)
persona_by_index = []         # parallel to `conversations`, used for the stratified cap below

with open(INPUT_DATA_FILE) as f:
    for line in f:
        conv = json.loads(line)
        conversations.append(conv)

        # Extract persona name from the "You are <Name>, ..." pattern
        sys_msg = conv["conversations"][0]["value"]
        match = re.match(r"You are (.+?),", sys_msg)
        if match:
            raw_name = match.group(1)
            # Normalize to a snake_case key: lowercase, strip leading "the ", spaces -> underscores
            key = raw_name.lower()
            key = re.sub(r"^the\s+", "", key)
            key = key.replace(" ", "_")
            persona_counts[key] += 1
            if key not in persona_system_prompts:
                persona_system_prompts[key] = sys_msg
        else:
            key = "unknown"
            print(f"  WARNING: could not extract persona from system prompt: {sys_msg[:80]}...")
        persona_by_index.append(key)

dataset = HFDataset.from_list(conversations)

# Apply the SFT cap from config — stratified per persona, deterministic seed. Applied AFTER
# loading the JSONL and BEFORE quality validation.
#
# Two properties this cell gets right, both of which a naive even split gets wrong:
#   1. WATER-FILLING. Taking min(target, available) per persona drops the shortfall from any
#      undersized persona on the floor - a requested 3000 quietly becomes 2361. The shortfall is
#      instead redistributed over personas that still have rows, repeating until the cap is met
#      or every pool is exhausted.
#   2. RANDOM SAMPLING. pool[:take] is a head slice, not a sample. The JSONL is grouped by topic,
#      so a head slice biases every capped run toward whichever topics datagen emitted first.
#      Seeding the RNG does not help if random is never called. Uses random.sample.
if SFT_MAX_EXAMPLES:
    if SFT_MAX_EXAMPLES < 0:
        raise ValueError(f"SFT_MAX_EXAMPLES must be 0 or positive, got {SFT_MAX_EXAMPLES}")
    random.seed(42)
    if len(dataset) > SFT_MAX_EXAMPLES:
        idx_by_persona = defaultdict(list)
        for i, p in enumerate(persona_by_index):
            idx_by_persona[p].append(i)

        # Water-fill: repeatedly split the remaining budget evenly across personas that still have
        # unallocated rows, capping each at what it actually has.
        personas_sorted = sorted(persona_counts.keys(), key=lambda p: -persona_counts[p])
        allocated = {p: 0 for p in personas_sorted}
        remaining = SFT_MAX_EXAMPLES
        open_personas = [p for p in personas_sorted if persona_counts[p] > 0]
        while remaining > 0 and open_personas:
            share = remaining // len(open_personas)
            if share == 0:
                # Fewer rows left than personas: hand out one each, largest pools first.
                for p in open_personas[:remaining]:
                    allocated[p] += 1
                    remaining -= 1
                break
            progressed = False
            for p in list(open_personas):
                take = min(share, persona_counts[p] - allocated[p])
                if take <= 0:
                    open_personas.remove(p)
                    continue
                allocated[p] += take
                remaining -= take
                progressed = True
                if allocated[p] >= persona_counts[p]:
                    open_personas.remove(p)
            if not progressed:
                break

        selected_idx = []
        for p in personas_sorted:
            selected_idx.extend(random.sample(idx_by_persona[p], allocated[p]))
        random.shuffle(selected_idx)

        even_target = SFT_MAX_EXAMPLES // len(personas_sorted)
        print(f"  Stratified cap: target {SFT_MAX_EXAMPLES} across {len(personas_sorted)} personas "
              f"(even split would be {even_target}/persona; shortfalls redistributed)")
        print(f"  Actually selected: {len(selected_idx)} rows")
        for p in personas_sorted:
            flag = "  <- pool exhausted" if allocated[p] >= persona_counts[p] else ""
            print(f"    {p:<25} {allocated[p]:>5} / {persona_counts[p]:<6} available{flag}")
        if len(selected_idx) < SFT_MAX_EXAMPLES:
            print(f"  NOTE: only {len(selected_idx)} rows available in total (< {SFT_MAX_EXAMPLES})")

        dataset = dataset.select(selected_idx)
        persona_counts = defaultdict(int, {p: allocated[p] for p in personas_sorted if allocated[p]})
    else:
        print(f"  SFT_MAX_EXAMPLES={SFT_MAX_EXAMPLES}, but only {len(dataset)} examples available; using all")

print(f"\n{'='*50}")
print(f"Total dataset: {len(dataset)} multi-turn conversations across {len(persona_counts)} personas")
print(f"Extracted {len(persona_system_prompts)} unique system prompts from JSONL")
print(f"Columns: {dataset.column_names}")
print(f"\nPer-persona breakdown:")
for p, c in sorted(persona_counts.items(), key=lambda x: -x[1]):
    print(f"  {p:20s} {c:>5d} conversations")

# Flag severe imbalance — a persona at a tiny share will not hold its voice after training, and
# that is worth surfacing before a multi-hour run rather than after.
_total = sum(persona_counts.values())
_weak = [(p, c) for p, c in persona_counts.items() if c / _total < 0.05]
if _weak:
    print(f"\n  WARNING: personas under 5% of the training mix — expect weak voice separation:")
    for p, c in sorted(_weak, key=lambda x: x[1]):
        print(f"    {p:<25} {c:>5} ({100 * c / _total:.1f}%)")

# Show a sample prompt to verify extraction
sample_key = next(iter(persona_system_prompts))
print(f"\n--- Sample extracted prompt ({sample_key}, first 200 chars) ---")
print(f"  {persona_system_prompts[sample_key][:200]}...")

LOADING COMBINED SHAREGPT DATA
  File: /workspace/training/biblical/data/training-data/biblical_persona_v2/biblical_personas_combined_sharegpt.jsonl

Total dataset: 4352 multi-turn conversations across 26 personas
Extracted 26 unique system prompts from JSONL
Columns: ['conversations', 'data_type']

Per-persona breakdown:
  moses                  785 conversations
  jeremiah               381 conversations
  paul                   377 conversations
  david                  350 conversations
  ezekiel                341 conversations
  isaiah                 332 conversations
  solomon                254 conversations
  job                    244 conversations
  daniel                 239 conversations
  peter                  148 conversations
  zechariah              145 conversations
  hosea                  107 conversations
  amos                    92 conversations
  joshua                  88 conversations
  micah                   73 conversations
  apostle_john            71 co

## 4. Validate & Summarize Dataset

Datagen output is already clean, so this is a cheap failure gate rather than a cleaning step —
bad data should fail here, in seconds, not two hours into training.

In [4]:
bad_examples = []
empty_responses = []
unique_system_prompts = set()

for i, example in enumerate(dataset):
    convs = example["conversations"]
    # Multi-turn ShareGPT: system, then alternating human/gpt pairs
    if len(convs) < 3 or len(convs) % 2 == 0:
        bad_examples.append((i, f"Expected odd turn count >=3, got {len(convs)}"))
        continue
    if convs[0]["from"] != "system":
        bad_examples.append((i, f"First turn should be 'system', got '{convs[0]['from']}'"))
        continue
    # Validate alternating human/gpt after system
    role_ok = True
    for j in range(1, len(convs)):
        expected = "human" if j % 2 == 1 else "gpt"
        if convs[j]["from"] != expected:
            bad_examples.append((i, f"Turn {j} should be '{expected}', got '{convs[j]['from']}'"))
            role_ok = False
            break
    if not role_ok:
        continue
    # Check the last GPT response is not empty
    if len(convs[-1]["value"].strip()) == 0:
        empty_responses.append(i)
    unique_system_prompts.add(convs[0]["value"])

# Turn-count distribution
from collections import Counter
turn_dist = Counter(len(ex["conversations"]) for ex in dataset)

print("DATA QUALITY CHECK")
print(f"  Total examples: {len(dataset)}")
print(f"  Bad structure: {len(bad_examples)}")
print(f"  Empty responses: {len(empty_responses)}")
print(f"  Unique system prompts: {len(unique_system_prompts)} "
      f"(should match extracted count: {len(persona_system_prompts)})")
print(f"  Turn distribution: {dict(sorted(turn_dist.items()))}")

if bad_examples:
    print(f"\nBad examples (first 5):")
    for idx, reason in bad_examples[:5]:
        print(f"    Example {idx}: {reason}")

# A malformed-structure rate this high means the datagen output changed shape. Training on it
# would silently teach the wrong turn structure, so stop here rather than at hour three.
if len(bad_examples) > 0.01 * max(len(dataset), 1):
    raise RuntimeError(
        f"{len(bad_examples)}/{len(dataset)} conversations have bad ShareGPT structure "
        f"(>1%). Regenerate the dataset before training."
    )

if empty_responses:
    print(f"\nFiltering {len(empty_responses)} empty responses...")
    _drop = set(empty_responses)
    dataset = dataset.select([i for i in range(len(dataset)) if i not in _drop])
    print(f"  Dataset after filtering: {len(dataset)} examples")

if len(dataset) == 0:
    raise RuntimeError("Dataset is empty after validation. Nothing to train on.")

# Persona distribution
print(f"\nPERSONA DISTRIBUTION:")
max_name_len = max(len(n) for n in persona_counts)
for name, count in sorted(persona_counts.items(), key=lambda x: -x[1]):
    bar = "#" * (count // 50)
    print(f"  {name:<{max_name_len}} {count:>5}  {bar}")
print(f"  {'TOTAL':<{max_name_len}} {sum(persona_counts.values()):>5}")

# Show voice differentiation — first response from a few different personas
print(f"\nVOICE SAMPLES (first ~100 chars of response):")
seen_personas = set()
for example in dataset:
    system = example["conversations"][0]["value"]
    name_part = system.split(",")[0].replace("You are ", "")
    if name_part not in seen_personas and len(seen_personas) < 4:
        response_start = example["conversations"][2]["value"][:100]
        print(f"  {name_part}: \"{response_start}...\"")
        seen_personas.add(name_part)

print(f"\nDataset validated and ready for training")

DATA QUALITY CHECK
  Total examples: 4352
  Bad structure: 0
  Empty responses: 0
  Unique system prompts: 26 (should match extracted count: 26)
  Turn distribution: {3: 1741, 5: 91, 7: 541, 9: 1979}

PERSONA DISTRIBUTION:
  moses          785  ###############
  jeremiah       381  #######
  paul           377  #######
  david          350  #######
  ezekiel        341  ######
  isaiah         332  ######
  solomon        254  #####
  job            244  ####
  daniel         239  ####
  peter          148  ##
  zechariah      145  ##
  hosea          107  ##
  amos            92  #
  joshua          88  #
  micah           73  #
  apostle_john    71  #
  james           54  #
  malachi         44  
  joel            44  
  zephaniah       37  
  habakkuk        32  
  jonah           29  
  nahum           27  
  haggai          24  
  obadiah         19  
  jude            15  
  TOTAL         4352

VOICE SAMPLES (first ~100 chars of response):
  Daniel: "Four is the number that stay

## 5. Load Model & Tokenizer (4-bit)

Loads `unsloth/Qwen3.5-4B` bf16 weights (~8.7 GB, already in the shared cache) and quantizes to
bnb NF4 on the fly.

This is a `*ForConditionalGeneration` checkpoint, so Unsloth returns a multimodal **Processor**,
not a tokenizer. The cell unwraps it — that is load-bearing, not cosmetic, and the comments below
say exactly what breaks without it.

In [5]:
import os
# DGX Spark (sm_120 / GB10): disable Unsloth's flex_attention override and the torch.compile path,
# both of which fail or fall into a slow eager path on this architecture. Carried over from the
# working Gemma 4 12B and Qwen3.8 27B runs on this machine. Must be set BEFORE `import unsloth`.
os.environ["UNSLOTH_ENABLE_FLEX_ATTENTION"] = "0"
os.environ["UNSLOTH_COMPILE_DISABLE"] = "1"

from unsloth import FastLanguageModel
import torch

# NOTE: no torch.cuda.set_per_process_memory_fraction() here, deliberately. On GB10 host and device
# share one 128 GB pool, so a fractional cap protects nothing — it only adds a ceiling that the
# bf16 GGUF merge at the end of this notebook would have to fit under. PYTORCH_CUDA_ALLOC_CONF is
# set in Cell 1 instead. See docs/dgx_spark_gb10_quirks.md.

model, tokenizer = FastLanguageModel.from_pretrained(
    model_name=BASE_LLM,
    max_seq_length=MAX_SEQ_LENGTH,
    load_in_4bit=True,
)

# ---- Multimodal base: unwrap the Processor -------------------------------------
# Qwen3.5-4B is Qwen3_5ForConditionalGeneration, so Unsloth loads it via AutoModelForVision2Seq
# and returns a multimodal Processor. Unwrap it and use the inner tokenizer for the rest of the
# notebook. Three concrete things this prevents:
#   - TRL routing. With a ProcessorMixin as processing_class, TRL marks the model vision-capable
#     and calls process_row(), which expects image inputs.
#   - Length grouping. Trainer._get_train_sampler reads processing_class.model_input_names[0]. On
#     a Processor that is "pixel_values", not "input_ids" -> ValueError before step 0.
#   - Calling convention. A Processor's first positional parameter is `images`, not `text`, so
#     tokenizer("some string") binds to the wrong argument.
# `processor` is kept so the adapter directory is saved with the same files the confirmed Qwen3.8
# adapters carry (processor_config.json alongside tokenizer_config.json / chat_template.jinja).
processor = None
if hasattr(tokenizer, "tokenizer"):
    processor = tokenizer
    tokenizer = processor.tokenizer
    print("  (Extracted tokenizer from Processor - text-only SFT mode)")

# Pad token. This checkpoint ships pad_token = '<|vision_pad|>' (248055), distinct from
# eos = '<|im_end|>' (248046). That is fine and is left alone — a pad token distinct from EOS is
# strictly better for causal-LM training. The fallback only fires if the tokenizer has no pad.
if tokenizer.pad_token is None:
    tokenizer.pad_token = tokenizer.eos_token
    print(f"  Tokenizer had no pad_token; set pad = eos ({tokenizer.eos_token!r})")

model.config.pad_token_id = tokenizer.pad_token_id
if getattr(model, "generation_config", None) is not None:
    model.generation_config.pad_token_id = tokenizer.pad_token_id

vocab_size = getattr(tokenizer, "vocab_size", "unknown")

print(f"Model loaded: {BASE_LLM}")
print(f"  Precision: 4-bit (QLoRA, quantized on load)")
print(f"  Max sequence length: {MAX_SEQ_LENGTH}")
print(f"  Vocab size: {vocab_size}")
print(f"  Tokenizer class: {type(tokenizer).__name__}"
      f"{' (unwrapped from ' + type(processor).__name__ + ')' if processor else ''}")
print(f"  pad_token: {tokenizer.pad_token!r} (id={tokenizer.pad_token_id})  "
      f"eos_token: {tokenizer.eos_token!r} (id={tokenizer.eos_token_id})")
print(f"  Attn impl: {getattr(model.config, '_attn_implementation', 'unknown')}")
print(f"  GPU allocated: {torch.cuda.memory_allocated()/1e9:.1f} GB")

==((====))==  Unsloth 2026.8.22: Fast Qwen3_5 patching. Transformers: 5.5.0.
   \\   /|    NVIDIA GB10. Num GPUs = 1. Max memory: 121.689 GB. Platform: Linux.
O^O/ \_/ \    Torch: 2.10.0a0+b558c986e8.nv25.11. CUDA: 12.1. CUDA Toolkit: 13.0. Triton: 3.4.0
\        /    Bfloat16 = TRUE. FA [Xformers = 0.0.33+aa7bc36.d20260302. FA2 = True]
 "-____-"     Free license: http://github.com/unslothai/unsloth
Unsloth: Fast downloading is enabled - ignore downloading bars which are red colored!


Loading weights:   0%|          | 0/723 [00:00<?, ?it/s]

  (Extracted tokenizer from Processor - text-only SFT mode)
Model loaded: unsloth/Qwen3.5-4B
  Precision: 4-bit (QLoRA, quantized on load)
  Max sequence length: 3072
  Vocab size: 248044
  Tokenizer class: TokenizersBackend (unwrapped from Qwen3VLProcessor)
  pad_token: '<|vision_pad|>' (id=248055)  eos_token: '<|im_end|>' (id=248046)
  Attn impl: flash_attention_2
  GPU allocated: 3.4 GB


## 6. Format Dataset for Chat Template

Apply Qwen's ChatML template to each conversation and build the training dataset.

Rendered with `add_generation_prompt=False` and `enable_thinking=False`. Three guards run
afterwards, each covering a failure that is silent for the whole run if it slips through:

1. **System prompts survived byte-identical** — the persona prompt is the entire point of this
   corpus, so a template that rewrites or prepends to it would poison every row.
2. **No unclosed `<think>` block** — training text must never open a reasoning block it does not
   close, or the model learns to emit one and never stop.
3. **Nothing truncates** — every rendered row is re-measured against `MAX_SEQ_LENGTH`.

The formatted dataset is cached to disk with a manifest keyed on the input file's SHA-256, the
tokenizer, and `MAX_SEQ_LENGTH`, so a kernel restart skips re-formatting. A content hash rather
than an mtime means an edited-in-place dataset can never be silently reused.

In [6]:
import hashlib, json, os
from pathlib import Path

from datasets import Dataset as HFDataset, load_from_disk
from unsloth.chat_templates import standardize_sharegpt

CACHE_VERSION = 1
_manifest_path = Path(FORMATTED_DATASET_DIR) / "manifest.json"
_dataset_path = Path(FORMATTED_DATASET_DIR) / "dataset"

original_dataset_len = len(dataset)

# Source system prompts, captured BEFORE standardize_sharegpt rewrites the role keys. Used by the
# byte-identity guard below.
_source_systems = [ex["conversations"][0]["value"] for ex in dataset]


def _file_sha256(path, chunk=1 << 20):
    h = hashlib.sha256()
    with open(path, "rb") as fh:
        for block in iter(lambda: fh.read(chunk), b""):
            h.update(block)
    return h.hexdigest()


_fingerprint = {
    "cache_version": CACHE_VERSION,
    "input_data_file": INPUT_DATA_FILE,
    "input_sha256": _file_sha256(INPUT_DATA_FILE),
    "tokenizer_class": type(tokenizer).__name__,
    "tokenizer_name": BASE_LLM,
    "tokenizer_vocab_size": int(getattr(tokenizer, "vocab_size", -1)),
    "max_seq_length": MAX_SEQ_LENGTH,
    "sft_max_examples": SFT_MAX_EXAMPLES,
    "enable_thinking_training": False,
    "row_count": original_dataset_len,
}

_cached = None
if not FORCE_REFORMAT and _manifest_path.exists() and _dataset_path.exists():
    try:
        _stored = json.loads(_manifest_path.read_text())
    except (OSError, json.JSONDecodeError):
        _stored = None
    if _stored == _fingerprint:
        _cached = load_from_disk(str(_dataset_path))
        print(f"Reusing formatted dataset cache: {_dataset_path} ({len(_cached)} rows)")
    elif _stored is not None:
        _diff = [k for k in _fingerprint if _stored.get(k) != _fingerprint[k]]
        print(f"Formatted-dataset cache is stale (changed: {_diff}) - rebuilding")

if _cached is not None:
    dataset = _cached
    filtered_dataset_len = len(dataset)
else:
    dataset = standardize_sharegpt(dataset)
    formatted_texts = tokenizer.apply_chat_template(
        list(dataset["conversations"]),
        tokenize=False,
        # Always False for TRAINING, regardless of ENABLE_THINKING. On this template that happens
        # to equal omitting the kwarg, but "happens to equal today" is not a contract.
        enable_thinking=False,
    )

    # ---- Guard 1: persona system prompts survived the template byte-identical ----
    # A template that trimmed, rewrote, or prepended an instruction to the system block would
    # poison every row in a persona corpus, and nothing downstream would notice.
    _corrupted = []
    for _i, (_text, _sys) in enumerate(zip(formatted_texts, _source_systems)):
        _expected = f"<|im_start|>system\n{_sys.strip()}<|im_end|>\n"
        if not _text.startswith(_expected):
            _corrupted.append(_i)
        if len(_corrupted) >= 3:
            break
    if _corrupted:
        _i = _corrupted[0]
        raise RuntimeError(
            f"{len(_corrupted)}+ examples do not open with their own system prompt verbatim "
            f"(first: row {_i}). The chat template altered or prefixed the system block - "
            f"rendered head: {formatted_texts[_i][:200]!r}. Do not start training."
        )

    # ---- Guard 2: no unclosed reasoning block ----
    # This template renders the final assistant turn as '<think>\n\n</think>\n\n' + content, which
    # matches what a non-thinking generation prompt produces - training and serving agree. What
    # must never appear is a <think> with content, or one that is never closed.
    _bad_think = sum(
        1 for _t in formatted_texts
        if _t.count("<think>") != _t.count("</think>") or "<think>\n\n</think>" not in _t
    )
    if _bad_think:
        raise RuntimeError(
            f"{_bad_think}/{len(formatted_texts)} formatted examples contain a reasoning block "
            "that is unclosed or non-empty. enable_thinking=False did not take effect - "
            "do not start training."
        )

    dataset = HFDataset.from_list([{"text": t} for t in formatted_texts])
    dataset = dataset.filter(lambda x: len(x["text"]) > 0)
    filtered_dataset_len = len(dataset)
    dataset = dataset.shuffle(seed=42)

    Path(FORMATTED_DATASET_DIR).mkdir(parents=True, exist_ok=True)
    dataset.save_to_disk(str(_dataset_path))
    _manifest_path.write_text(json.dumps(_fingerprint, indent=2))
    print(f"Formatted dataset cached to {_dataset_path}")

# ---- Guard 3: nothing truncates ----
# MAX_SEQ_LENGTH was set from a measurement of this corpus. Re-measure rather than trust it: a
# regenerated dataset with longer answers would be silently cut mid-response.
_lengths = [len(tokenizer(t, add_special_tokens=False)["input_ids"]) for t in dataset["text"]]
_over = sum(1 for n in _lengths if n > MAX_SEQ_LENGTH)
if _over:
    raise RuntimeError(
        f"{_over}/{len(_lengths)} formatted examples exceed MAX_SEQ_LENGTH={MAX_SEQ_LENGTH} "
        f"(longest is {max(_lengths)} tokens) and would be truncated mid-response. "
        f"Raise MAX_SEQ_LENGTH in Cell 1 and re-run."
    )

import numpy as np
_a = np.array(_lengths)
print(f"\n--- Sample formatted text (first 500 chars) ---")
print(dataset[0]["text"][:500])
print(f"\nDataset formatted: {len(dataset)} examples")
print(f"  Original conversations: {original_dataset_len}")
print(f"  After empty filtering:  {filtered_dataset_len}")
print(f"  SFT cap:                {SFT_MAX_EXAMPLES or 'ALL'}")
print(f"  Token lengths:          p50={int(np.percentile(_a, 50))}  p90={int(np.percentile(_a, 90))}  "
      f"p99={int(np.percentile(_a, 99))}  max={int(_a.max())}  (limit {MAX_SEQ_LENGTH})")
print(f"  Guards passed:          system prompts verbatim, no open <think>, no truncation")

Reusing formatted dataset cache: /workspace/training/biblical/output/biblical_qwen3_5_4b_unsloth_4bit/train/formatted_sft_dataset_cache/dataset (4352 rows)

--- Sample formatted text (first 500 chars) ---
<|im_start|>system
You are David, the king of Israel, once a shepherd boy and warrior, the sweet psalmist who poured out your heart in song, a man after God's own heart who knew both triumph and deep sin.

YOUR DISTINCTIVE VOICE: Poetic, emotional, lyrical. Psalm cadence — parallelism, repetition, selah-like pauses. Shifts between ecstatic praise, raw anguish, and intimate confession. Uses nature imagery (shepherd, waters, mountains). Addresses God directly ('O LORD'). Unashamed emotional vulner

Dataset formatted: 4352 examples
  Original conversations: 4352
  After empty filtering:  4352
  SFT cap:                ALL
  Token lengths:          p50=1575  p90=2138  p99=2494  max=2999  (limit 3072)
  Guards passed:          system prompts verbatim, no open <think>, no truncation


## 7. Add LoRA Adapters

### Scoping the adapter to the language model — deliberately, not by accident

`unsloth/Qwen3.5-4B` is a multimodal checkpoint: alongside the 32-layer language model it carries a
24-block vision tower (`model.visual.*`, 297 tensors) and a 1-layer MTP head (`mtp.*`, 15 tensors).
This notebook trains a **text-only** persona LoRA, so neither should receive an adapter.

That does **not** happen for free. PEFT matches a `target_modules` list like `["q_proj", "k_proj", ...]`
by **suffix, across the entire module tree** — it has no notion of "the language model". Whether a
sub-model gets caught depends purely on whether it reuses those leaf names. What that means *on this
specific checkpoint and stack*, verified rather than assumed:

| Sub-model | Leaf names | Reached by a bare list? |
|---|---|---|
| Vision tower | `visual.blocks.N.attn.qkv`, `attn.proj`, `mlp.linear_fc1/linear_fc2` | **No** — no collision. By luck of naming, not by design. |
| MTP head | `mtp.layers.0.self_attn.{q,k,v,o}_proj`, `mtp.layers.0.mlp.{gate,up,down}_proj` | **Names collide, but not reachable today.** `transformers 5.16.0.dev0` builds only `model.visual`, `model.language_model` and `lm_head` for this architecture — checked directly, the loaded tree has **0** `mtp` parameters. |

So on *this* stack a bare list would happen to come out clean. That is a property of the installed
`transformers` version, not of the checkpoint — the same bare list on Qwen3.8-27B attaches 7 adapters
to `mtp.*`, and on Gemma 4 it attaches 224 vision + 72 audio tensors (two adapters already shipped in
this workspace that way, 36–42% dead weight, and **vLLM refuses to load an adapter carrying
vision-layer LoRA**). Relying on a module simply not being instantiated is not scoping.

The `finetune_*` flags below make the scoping explicit and version-independent. They are load-bearing:
Unsloth routes an explicit list through its `language|text` scoping regex **only when at least one
flag is False**, and all four default to True. With them set, Unsloth itself confirms the constraint
on stdout:

```
Unsloth: Explicit target_modules are constrained by the finetune_(vision|language|attention|mlp)
filters; adapters attach only where both select.
```

**Verified result on this machine** — the cell below resolves to exactly **128 modules**: 96 language
MLP (`gate`/`up`/`down` × all 32 layers) + 32 language `self_attn` (`q`/`k`/`v`/`o` × layers
3, 7, 11, 15, 19, 23, 27, 31 — precisely the 8 `full_attention` layers). Vision tower: 297 params,
frozen. Trainable: 42,467,328 / 2,646,192,640 (1.60%).

The assertions after the call prove that held for *your* run rather than trusting this paragraph.


In [7]:
model = FastLanguageModel.get_peft_model(
    model,
    r=LORA_R,
    target_modules=LORA_TARGET_MODULES,
    lora_alpha=LORA_ALPHA,
    lora_dropout=LORA_DROPOUT,
    bias="none",
    # ---- SCOPE FREEZE: deliberate, not incidental ----------------------------------
    # These four flags are load-bearing, not decoration. Unsloth routes an explicit
    # target_modules list through its language-scoped regex ONLY when at least one of them is
    # False; all four default to True, so a bare list goes straight to PEFT, which matches by
    # SUFFIX across the WHOLE module tree. finetune_vision_layers=False is what confines the match
    # to model.language_model.*, independent of which sub-models the installed transformers
    # happens to instantiate. See this cell's markdown for what that does and does not reach here.
    finetune_vision_layers=False,
    finetune_language_layers=True,
    finetune_attention_modules=True,
    finetune_mlp_modules=True,
    # Deliberately True, NOT "unsloth". VERIFIED in unsloth_zoo/gradient_checkpointing.py: the
    # "unsloth" path copies saved activations into PINNED host buffers
    # (torch.empty(..., device="cpu", pin_memory=True)) and only ever grows them
    # (`if new_size > x.numel(): x.resize_(new_size)`); the sole shrink path runs at teardown,
    # never mid-run. On a discrete GPU that trade is a win - VRAM is scarce, host RAM is not. On
    # GB10 host and device are the SAME 128 GB pool, so the copy frees nothing while ratcheting
    # unreclaimable page-locked memory upward for hours. True uses standard recompute
    # checkpointing with no host copies.
    use_gradient_checkpointing=True,
    random_state=3407,
    max_seq_length=MAX_SEQ_LENGTH,
)

# ============ VERIFY THE ADAPTER SCOPE AND THE TOWER/MTP FREEZE ============
# Two independent properties, checked separately because they fail in different ways:
#   1. WHERE ADAPTERS ATTACHED. If the finetune_* flags did not take effect, the bare leaf-name
#      list reaches PEFT and matches by suffix across the whole model, adapting mtp.*. vLLM
#      rejects an adapter carrying those keys.
#   2. WHAT IS TRAINABLE. Under QLoRA base weights are frozen 4-bit and only LoRA A/B carry
#      gradients, so the vision tower and MTP head are frozen by construction. That is exactly
#      why it is worth asserting - an assumption never checked is one that eventually breaks.
# Both raise. A silent version of either burns the entire run. See
# docs/multimodal_and_hybrid_base_models.md.
from collections import Counter


def _zone_of(param_name):
    """Return the non-language sub-model a parameter belongs to, or None."""
    n = param_name.replace("base_model.model.", "", 1)
    if n.startswith("mtp.") or ".mtp." in n:
        return "mtp head"
    if ".visual." in n or n.startswith("visual."):
        return "vision tower"
    if "vision_tower" in n:
        return "vision tower"
    if "audio_tower" in n:
        return "audio tower"
    if "multi_modal_projector" in n or "embed_vision" in n or "embed_audio" in n:
        return "mm projector"
    return None


_adapted = sorted({
    n.split(".lora_A")[0].split(".lora_B")[0].replace("base_model.model.", "")
    for n, _ in model.named_parameters() if ".lora_A." in n or ".lora_B." in n
})

_families = Counter()
for n in _adapted:
    zone = _zone_of(n)
    if zone is not None:
        _families[f"{zone} (SHOULD BE 0)"] += 1
    elif ".self_attn." in n:
        _families["language self_attn"] += 1
    elif ".mlp." in n:
        _families["language mlp"] += 1
    else:
        _families[f"other: {n}"] += 1

print(f"LoRA adapters added (r={LORA_R}, alpha={LORA_ALPHA})")
print(f"  Adapted modules: {len(_adapted)}")
for fam, count in sorted(_families.items()):
    print(f"    {fam:<30} {count}")

# --- Check 1: adapter placement ---
_stray = [n for n in _adapted if _zone_of(n)]
if _stray:
    raise RuntimeError(
        f"LoRA attached to {len(_stray)} module(s) outside the language model "
        f"(e.g. {_stray[:3]}). The finetune_* scoping flags did not take effect. "
        "Do not start training - vLLM refuses adapters carrying vision/MTP LoRA."
    )

# --- Check 2: vision tower and MTP head are frozen ---
_unfrozen, _zone_params = {}, Counter()
for _n, _p in model.named_parameters():
    _z = _zone_of(_n)
    if _z is None:
        continue
    _zone_params[_z] += 1
    if _p.requires_grad:
        _unfrozen.setdefault(_z, []).append(_n)

for _z, _c in sorted(_zone_params.items()):
    _bad = len(_unfrozen.get(_z, []))
    print(f"  {_z:<14} {_c:>5} params  "
          f"{'FROZEN' if not _bad else str(_bad) + ' TRAINABLE - BAD'}")
if _unfrozen:
    _sample = [n for v in _unfrozen.values() for n in v][:5]
    raise RuntimeError(
        f"{sum(len(v) for v in _unfrozen.values())} tower/MTP parameter(s) are trainable "
        f"(e.g. {_sample}). This is a text-only fine-tune - they must stay frozen."
    )

# --- Check 3: the count matches what this architecture produced when verified ---
# 128 = 96 MLP (gate/up/down x all 32 layers) + 32 self_attn (q/k/v/o x the 8 full_attention
# layers: 3, 7, 11, 15, 19, 23, 27, 31). This was confirmed by loading this exact base on this
# machine. A different number means the scoping regex resolved differently than it did then -
# worth a look before committing hours to the run.
_expected_modules = 128
if len(_adapted) != _expected_modules:
    print(f"  NOTE: expected {_expected_modules} adapted modules for this architecture "
          f"(96 MLP + 32 self_attn), got {len(_adapted)}. Not fatal - the placement and freeze "
          f"checks above already passed - but confirm the scoping regex resolved as intended "
          f"before starting a long run.")

trainable = sum(p.numel() for p in model.parameters() if p.requires_grad)
total = sum(p.numel() for p in model.parameters())
print(f"  Trainable parameters: {trainable:,} / {total:,} ({100*trainable/total:.4f}%)")
print(f"  Note: the 24 gated-delta linear_attn layers are not adapted by design "
      f"(in_proj_qkv / in_proj_a / in_proj_b / in_proj_z / out_proj leaf names); their MLPs are.")

Unsloth: Explicit target_modules are constrained by the finetune_(vision|language|attention|mlp) filters; adapters attach only where both select.
LoRA adapters added (r=32, alpha=32)
  Adapted modules: 128
    language mlp                   96
    language self_attn             32
  vision tower     297 params  FROZEN
  Trainable parameters: 42,467,328 / 2,646,192,640 (1.6048%)
  Note: the 24 gated-delta linear_attn layers are not adapted by design (in_proj_qkv / in_proj_a / in_proj_b / in_proj_z / out_proj leaf names); their MLPs are.


## 8. Trainer Setup

Loss is computed on **assistant responses only**. Without that mask it averages in the ~1,600-character
persona system prompt and the user turns — tokens the base model already predicts near-perfectly and
that repeat across every row — which pins the average near their own ~0 loss and buries the signal
from the response tokens. An unmasked Biblical run started at loss 2.23 and flattened near 0.9; a
masked run started at 7.32 and fell to 0.61. A high starting loss is the correct behavior here.

In [8]:
import gc
from transformers import TrainerCallback
from trl import SFTTrainer, SFTConfig


class PeriodicMemoryCleanup(TrainerCallback):
    """Collect dead Python objects and return cached CUDA blocks to the allocator.

    Runs every `every` optimizer steps. This does not repair a leak - it releases cached and
    fragmented blocks that would otherwise sit reserved but unused, which matters on GB10 where
    host and device draw from the same 128 GB pool. Deliberately on_step_end only, on an
    interval: empty_cache() forces the allocator to re-acquire blocks, so doing it twice a step
    measurably slows training and buys nothing.
    """

    def __init__(self, every=50):
        self.every = max(1, int(every))
        self.peak_reserved = 0.0

    def on_step_end(self, args, state, control, **kwargs):
        if state.global_step % self.every == 0:
            gc.collect()
            torch.cuda.empty_cache()
            # Report after the flush, so this is memory the allocator could NOT release.
            # On GB10 host and device share one 128 GB pool, so a reserved figure that keeps
            # climbing across these reports is the early signal of the run walking into an OOM -
            # the logits tensor is batch x seq x 248320 and every batch is a different shape.
            # Seeing it at step 50 is worth a great deal more than discovering it at step 900.
            reserved = torch.cuda.memory_reserved() / 1e9
            self.peak_reserved = max(self.peak_reserved, reserved)
            print(f"    [step {state.global_step}] reserved={reserved:.1f} GB  "
                  f"peak_alloc={torch.cuda.max_memory_allocated() / 1e9:.1f} GB")
            if reserved > 60:
                print("    WARNING: over 60 GB reserved on a 128 GB unified pool, and this "
                      "number should be roughly FLAT across reports, not climbing. If it is "
                      "climbing, something is stopping the allocator reusing blocks - check "
                      "that Cell 1 sets no PYTORCH_CUDA_ALLOC_CONF (see the note there), then "
                      "lower BATCH_SIZE.")
        return control


trainer = SFTTrainer(
    model=model,
    processing_class=tokenizer,
    train_dataset=dataset,
    args=SFTConfig(
        dataset_text_field="text",
        # `max_length`, not `max_seq_length`: TRL renamed it, and the installed TRL is 0.24.0.
        max_length=MAX_SEQ_LENGTH,
        # Explicitly False, and printed accurately below. Qwen3.5-4B is a hybrid linear-attention
        # model (24 gated-delta layers); its recurrent state and causal conv1d leak across
        # sequence boundaries once packing flattens a batch, so Unsloth force-disables packing for
        # it regardless of the flag. Setting True here would be a silent lie about what ran.
        packing=False,
        # NOTE: train_sampling_strategy="group_by_length" is deliberately absent - it does not
        # work with this model. Two independent blockers, both in transformers source:
        #   1. Trainer._get_train_sampler passes processing_class.model_input_names[0] to
        #      LengthGroupedSampler. Even with the Processor unwrapped, this path is fragile on
        #      Conditional-Generation models.
        #   2. Supplying a precomputed "length" column does not help: _remove_unused_columns
        #      prunes every column outside the model forward signature, and it runs BEFORE
        #      sampler_fn in _get_dataloader, so the column is gone by sampling time.
        # The default random sampler is used; batches still pad to their own longest member.
        per_device_train_batch_size=BATCH_SIZE,
        gradient_accumulation_steps=GRAD_ACCUM,
        warmup_steps=5,
        num_train_epochs=TARGET_EPOCHS,
        learning_rate=LEARNING_RATE,
        fp16=not torch.cuda.is_bf16_supported(),
        bf16=torch.cuda.is_bf16_supported(),
        logging_steps=5,
        optim="adamw_8bit",
        # Pinned host memory is page-locked and unreclaimable. On GB10 it comes out of the same
        # pool the model is training in, so the usual "pin for faster H2D" trade does not apply.
        dataloader_pin_memory=False,
        weight_decay=0.01,
        lr_scheduler_type="cosine",
        seed=3407,
        output_dir=OUTPUT_DIR_ADAPTERS,
        save_strategy="steps",
        save_steps=SAVE_STEPS,
        save_total_limit=SAVE_TOTAL_LIMIT,
        report_to="none",
    ),
    callbacks=[PeriodicMemoryCleanup(CLEANUP_STEPS)],
)

# ===================== TRAIN ON RESPONSES ONLY =====================
# Both marker args are left as None so Unsloth auto-detects them from the Qwen chat template and
# prints what it found - no hand-written marker strings to get wrong.
from unsloth.chat_templates import train_on_responses_only

trainer = train_on_responses_only(trainer)

# Verify the markers actually matched. If they do not, every label becomes -100, the run trains on
# nothing, and it fails silently for its entire duration. This must raise, not warn.
import numpy as np

_probe = trainer.train_dataset[:8]["labels"]
_kept = sum(int((np.array(x) != -100).sum()) for x in _probe)
_total_labels = sum(len(x) for x in _probe)
if _kept == 0:
    raise RuntimeError(
        "train_on_responses_only masked EVERY token - the instruction/response markers did not "
        "match the chat template. Do not start training."
    )
# The opposite failure: masking that silently did nothing. With ~1,600-char system prompts plus
# user turns, the assistant share of a row is well under 90%; anything at or above that means the
# prompt was not masked.
_kept_frac = _kept / max(_total_labels, 1)
if _kept_frac > 0.9:
    raise RuntimeError(
        f"Response masking kept {100 * _kept_frac:.1f}% of label tokens - far too high for this "
        "corpus. The prompt is not being masked; check the auto-detected markers above."
    )
print(f"Response masking OK: {_kept:,}/{_total_labels:,} label tokens kept "
      f"({100 * _kept_frac:.1f}%) across 8 sample sequences")

effective_batch_size = BATCH_SIZE * GRAD_ACCUM
_steps_per_epoch = -(-len(dataset) // effective_batch_size)
print(f"Trainer configured")
print(f"  Effective batch size: {BATCH_SIZE} x {GRAD_ACCUM} = {effective_batch_size}")
print(f"  Epochs: {TARGET_EPOCHS}  |  Steps: ~{_steps_per_epoch * TARGET_EPOCHS}")
print(f"  LR: {LEARNING_RATE} (cosine, {5} warmup steps)")
print(f"  Packing: disabled (hybrid linear-attention model)")
print(f"  Length grouping: disabled")
print(f"  processing_class: {type(tokenizer).__name__}"
      f"{' (Processor unwrapped at load)' if processor else ''}")
print(f"  Checkpoints: every {SAVE_STEPS} steps, keep {SAVE_TOTAL_LIMIT}")
print(f"  Memory cleanup: every {CLEANUP_STEPS} steps")
print(f"  Loss computed on: assistant responses only (prompt masked to -100)")
print(f"  Dataset: {len(dataset)} examples")
print(f"  Precision: {'bf16' if torch.cuda.is_bf16_supported() else 'fp16'}")

Unsloth: Auto-detected instruction_part = '\n<|im_start|>user\n' and response_part = '\n<|im_start|>assistant\n'
Response masking OK: 7,103/10,998 label tokens kept (64.6%) across 8 sample sequences
Trainer configured
  Effective batch size: 2 x 4 = 8
  Epochs: 1  |  Steps: ~544
  LR: 0.0002 (cosine, 5 warmup steps)
  Packing: disabled (hybrid linear-attention model)
  Length grouping: disabled
  processing_class: TokenizersBackend (Processor unwrapped at load)
  Checkpoints: every 50 steps, keep 3
  Memory cleanup: every 50 steps
  Loss computed on: assistant responses only (prompt masked to -100)
  Dataset: 4352 examples
  Precision: bf16


## 9. Train

Auto-resumes from the newest `checkpoint-*` in the train directory. The training kernel lives in
the container, so a running cell survives VS Code quitting, the browser closing, and SSH dropping;
durability comes from the container's `unless-stopped` policy plus this resume.

In [9]:
# Start training. Auto-resumes from the newest checkpoint in OUTPUT_DIR_ADAPTERS if one exists, so
# an interrupted run continues instead of restarting from step 0.
import os
from transformers.trainer_utils import get_last_checkpoint

_ckpt_dir = trainer.args.output_dir
last_checkpoint = get_last_checkpoint(_ckpt_dir) if os.path.isdir(_ckpt_dir) else None

if last_checkpoint:
    print(f"Resuming from checkpoint: {last_checkpoint}")
    result = trainer.train(resume_from_checkpoint=last_checkpoint)
else:
    print("No checkpoint found - starting from scratch.")
    result = trainer.train()

print("\nTraining complete")
print(f"  Final loss:  {result.training_loss:.4f}")
print(f"  Total steps: {result.global_step}")

The tokenizer has new PAD/BOS/EOS tokens that differ from the model config and generation config. The model config and generation config were aligned accordingly, being updated with the tokenizer's values. Updated tokens: {'eos_token_id': 248046}.
The tokenizer has new PAD/BOS/EOS tokens that differ from the model config and generation config. The model config and generation config were aligned accordingly, being updated with the tokenizer's values. Updated tokens: {'eos_token_id': 248046}.


No checkpoint found - starting from scratch.


==((====))==  Unsloth - 2x faster free finetuning | Num GPUs used = 1
   \\   /|    Num examples = 4,352 | Num Epochs = 1 | Total steps = 544
O^O/ \_/ \    Batch size per device = 2 | Gradient accumulation steps = 4
\        /    Data Parallel GPUs = 1 | Total batch size (2 x 4 x 1) = 8
 "-____-"     Trainable parameters = 42,467,328 of 4,581,732,864 (0.93% trained)
==((====))==  Unsloth - 2x faster free finetuning | Num GPUs used = 1
   \\   /|    Num examples = 4,352 | Num Epochs = 1 | Total steps = 544
O^O/ \_/ \    Batch size per device = 2 | Gradient accumulation steps = 4
\        /    Data Parallel GPUs = 1 | Total batch size (2 x 4 x 1) = 8
 "-____-"     Trainable parameters = 42,467,328 of 4,581,732,864 (0.93% trained)
Unsloth: Not an error, but Qwen3_5ForConditionalGeneration does not accept `num_items_in_batch`.
Using gradient accumulation will be very slightly less accurate.
Read more on gradient accumulation issues here: https://unsloth.ai/blog/gradient
Unsloth: Not an err

Step,Training Loss
5,2.143298
10,2.026032
15,1.753214
20,1.740781
25,1.641860
30,1.941821
35,1.822412
40,1.813687
45,1.872421
50,1.628353


    [step 50] reserved=3.7 GB  peak_alloc=12.4 GB


...

    [step 450] reserved=3.7 GB  peak_alloc=13.8 GB


Unsloth: Restored added_tokens_decoder metadata in /workspace/training/biblical/output/biblical_qwen3_5_4b_unsloth_4bit/train/checkpoint-450/tokenizer_config.json.
Unsloth: Restored added_tokens_decoder metadata in /workspace/training/biblical/output/biblical_qwen3_5_4b_unsloth_4bit/train/checkpoint-450/tokenizer_config.json.


    [step 500] reserved=3.7 GB  peak_alloc=13.8 GB


Unsloth: Restored added_tokens_decoder metadata in /workspace/training/biblical/output/biblical_qwen3_5_4b_unsloth_4bit/train/checkpoint-500/tokenizer_config.json.
Unsloth: Restored added_tokens_decoder metadata in /workspace/training/biblical/output/biblical_qwen3_5_4b_unsloth_4bit/train/checkpoint-500/tokenizer_config.json.
Unsloth: Restored added_tokens_decoder metadata in /workspace/training/biblical/output/biblical_qwen3_5_4b_unsloth_4bit/train/checkpoint-544/tokenizer_config.json.
Unsloth: Restored added_tokens_decoder metadata in /workspace/training/biblical/output/biblical_qwen3_5_4b_unsloth_4bit/train/checkpoint-544/tokenizer_config.json.



Training complete
  Final loss:  1.6775
  Total steps: 544


## 10. Save LoRA Adapters

Saves the adapter, the Processor (a superset of the tokenizer files), the persona system prompts,
and a machine-readable `complete.json` sentinel. The adapter loads on any quantization of the same
Qwen3.5-4B base via PEFT or vLLM.

In [10]:
from pathlib import Path
from datetime import datetime, timezone
import json

Path(LORA_OUTPUT_DIR).mkdir(parents=True, exist_ok=True)

print(f"Saving LoRA adapters to {LORA_OUTPUT_DIR}...")
model.save_pretrained(LORA_OUTPUT_DIR)
# Save the Processor when one was unwrapped, so the adapter directory carries the same files the
# confirmed Qwen3.8 adapters do (processor_config.json alongside tokenizer_config.json /
# chat_template.jinja). Saving a Processor also writes the tokenizer files, so this is a superset
# of tokenizer.save_pretrained().
(processor or tokenizer).save_pretrained(LORA_OUTPUT_DIR)

# Save system prompts alongside the adapter for inference use
prompts_path = f"{LORA_OUTPUT_DIR}/persona_system_prompts.json"
with open(prompts_path, "w") as f:
    json.dump(persona_system_prompts, f, indent=2)

# Machine-readable completion sentinel (docs/README.md: "Final model-producing notebooks must
# write a machine-readable completion sentinel"). The DPO notebook reads this to confirm the SFT
# stage it resumes from actually finished.
sentinel = {
    "status": "complete",
    "stage": "sft",
    "model_name": MODEL_NAME_BASE,
    "base_model": BASE_LLM,
    "input_data_file": INPUT_DATA_FILE,
    "output_dir": OUTPUT_DIR_ADAPTERS,
    "lora_output_dir": LORA_OUTPUT_DIR,
    "last_checkpoint": last_checkpoint,
    "global_step": int(result.global_step),
    "final_loss": float(result.training_loss),
    "num_train_epochs": TARGET_EPOCHS,
    "sft_max_examples": SFT_MAX_EXAMPLES,
    "train_examples": len(dataset),
    "max_seq_length": MAX_SEQ_LENGTH,
    "batch_size": BATCH_SIZE,
    "grad_accum": GRAD_ACCUM,
    "learning_rate": LEARNING_RATE,
    "lora_r": LORA_R,
    "lora_alpha": LORA_ALPHA,
    "lora_target_modules": LORA_TARGET_MODULES,
    "adapted_modules": len(_adapted),
    "formatted_dataset_fingerprint": _fingerprint,
    "personas": sorted(persona_system_prompts),
    "timestamp": datetime.now(timezone.utc).strftime("%Y-%m-%dT%H:%M:%SZ"),
}
with open(f"{LORA_OUTPUT_DIR}/complete.json", "w") as f:
    json.dump(sentinel, f, indent=2)

example_persona = next(iter(persona_system_prompts), "daniel")

print(f"\nLoRA adapters saved")
print(f"  Adapters:       {LORA_OUTPUT_DIR}")
print(f"  Sentinel:       {LORA_OUTPUT_DIR}/complete.json "
      f"(step {result.global_step}, loss {result.training_loss:.4f})")
print(f"  System prompts: {prompts_path} ({len(persona_system_prompts)} personas)")
print(f"\n  At inference, load prompts with:")
print(f'    with open("{prompts_path}") as f:')
print(f'        prompts = json.load(f)')
print(f'    system_msg = prompts["{example_persona}"]  # or any persona key')
print(f"\n  Audit the saved adapter for scope leakage:")
print(f"    docker exec unsloth-notebook python /workspace/training/docs/audit_adapters.py")

Saving LoRA adapters to /workspace/training/biblical/output/biblical_qwen3_5_4b_unsloth_4bit/lora_adapters...


Unsloth: Restored added_tokens_decoder metadata in /workspace/training/biblical/output/biblical_qwen3_5_4b_unsloth_4bit/lora_adapters/tokenizer_config.json.
Unsloth: Restored added_tokens_decoder metadata in /workspace/training/biblical/output/biblical_qwen3_5_4b_unsloth_4bit/lora_adapters/tokenizer_config.json.



LoRA adapters saved
  Adapters:       /workspace/training/biblical/output/biblical_qwen3_5_4b_unsloth_4bit/lora_adapters
  Sentinel:       /workspace/training/biblical/output/biblical_qwen3_5_4b_unsloth_4bit/lora_adapters/complete.json (step 544, loss 1.6775)
  System prompts: /workspace/training/biblical/output/biblical_qwen3_5_4b_unsloth_4bit/lora_adapters/persona_system_prompts.json (26 personas)

  At inference, load prompts with:
    with open("/workspace/training/biblical/output/biblical_qwen3_5_4b_unsloth_4bit/lora_adapters/persona_system_prompts.json") as f:
        prompts = json.load(f)
    system_msg = prompts["daniel"]  # or any persona key

  Audit the saved adapter for scope leakage:
    docker exec unsloth-notebook python /workspace/training/docs/audit_adapters.py


## 11. Test Inference

Smoke test with a few personas using their extracted system prompts. Each should answer in its own
voice.

`enable_thinking` is passed explicitly and comes from `ENABLE_THINKING` in Cell 1 — the eval must
use the same value the training data was built with, or the smoke test is measuring a prompt shape
the model never saw.

In [11]:
from transformers import TextStreamer

FastLanguageModel.for_inference(model)

# Pick up to 4 personas to test
test_personas = list(persona_system_prompts.keys())[:4]

print(f"INFERENCE TEST - {len(test_personas)} PERSONAS (thinking={'ON' if ENABLE_THINKING else 'OFF'})\n")

for persona_key in test_personas:
    system_prompt = persona_system_prompts[persona_key]

    messages = [
        {"role": "system", "content": system_prompt},
        {"role": "user", "content": TEST_PROMPT},
    ]

    text = tokenizer.apply_chat_template(
        messages,
        tokenize=False,
        add_generation_prompt=True,
        enable_thinking=ENABLE_THINKING,
    )
    # Passed by keyword: harmless for a tokenizer, and required if this ever runs against a
    # Processor, whose first positional parameter is `images`, not `text`.
    inputs = tokenizer(text=text, return_tensors="pt").to(model.device)

    print(f"{'='*60}")
    print(f"  PERSONA: {persona_key.upper()}")
    print(f"  Q: {TEST_PROMPT}")
    print(f"  A: ", end="")

    outputs = model.generate(
        **inputs,
        max_new_tokens=1024 if ENABLE_THINKING else 384,
        temperature=GEN_TEMPERATURE,
        top_p=GEN_TOP_P,
        top_k=GEN_TOP_K,
        do_sample=True,
        streamer=TextStreamer(tokenizer, skip_prompt=True),
    )
    print()

del inputs, outputs

INFERENCE TEST - 4 PERSONAS (thinking=OFF)

  PERSONA: DANIEL
  Q: I am struggling with forgiveness. What does Scripture teach about forgiving others?
  A: In the third year of King Belshazzar, I saw a ram with two horns, one higher than the other, and the high one grew great, but then a goat came with a horn between its eyes, and it attacked the ram and cast it to the ground—yet I knew that which was written, for the Most High does not endure in the pride of men.  

So too with forgiveness: it is not a matter of our strength, but of His who dwells in eternity. I have stood before kings who feared the gods of Babylon more than they feared their own souls; I have seen men crushed beneath the weight of their sins, their faces buried in dust, their hearts unyielding. But I have also seen the God of Israel raise up a horn that should break the power of the horn of the ram—the same God who shut the lions’ mouths and preserved my life in the den.  

Forgiveness is not a release from judgment

## 12. Verify Adapter (Cold Reload from Disk)

Final validation: load the adapter cold from disk to confirm it is self-contained and portable.
The notebook is not finished until this passes.

In [12]:
# Clean up the training model
import gc, torch
del model, tokenizer, trainer, dataset
gc.collect()
torch.cuda.empty_cache()

print("Cleared training model from memory")
print(f"  Loading adapter from: {LORA_OUTPUT_DIR}")

# Reload from disk
model2, tokenizer2 = FastLanguageModel.from_pretrained(
    model_name=LORA_OUTPUT_DIR,
    max_seq_length=MAX_SEQ_LENGTH,
    load_in_4bit=True,
)
FastLanguageModel.for_inference(model2)

if hasattr(tokenizer2, "tokenizer"):
    tokenizer2 = tokenizer2.tokenizer

# Reload saved system prompts + sentinel
import json
from pathlib import Path
with open(f"{LORA_OUTPUT_DIR}/persona_system_prompts.json") as f:
    reloaded_prompts = json.load(f)
with open(f"{LORA_OUTPUT_DIR}/complete.json") as f:
    reloaded_sentinel = json.load(f)

# Confirm from the SAVED ARTIFACT, not from this notebook's variables, that the adapter carries
# only language-model modules. adapter_config.json and the safetensors keys are ground truth;
# a notebook drifts from what produced its output.
import struct
from collections import Counter
_sf = Path(LORA_OUTPUT_DIR) / "adapter_model.safetensors"
with open(_sf, "rb") as f:
    _hdr = json.loads(f.read(struct.unpack("<Q", f.read(8))[0]))
_fam = Counter()
for k in _hdr:
    if k == "__metadata__":
        continue
    n = k.replace("base_model.model.", "")
    _fam["VISION" if ("vision_tower" in n or ".visual." in n)
         else "AUDIO" if "audio_tower" in n
         else "MTP" if (n.startswith("mtp.") or ".mtp." in n)
         else "language"] += 1
print(f"\nSaved adapter tensor families: {dict(_fam)}")
if set(_fam) != {"language"}:
    raise RuntimeError(
        f"Saved adapter contains non-language tensors: {dict(_fam)}. "
        "vLLM will refuse this adapter. Do not ship it."
    )

# Test with the first persona
test_key = list(reloaded_prompts.keys())[0]
test_prompt_text = reloaded_prompts[test_key]

messages = [
    {"role": "system", "content": test_prompt_text},
    {"role": "user", "content": TEST_PROMPT},
]

text = tokenizer2.apply_chat_template(
    messages,
    tokenize=False,
    add_generation_prompt=True,
    enable_thinking=ENABLE_THINKING,
)
inputs = tokenizer2(text=text, return_tensors="pt").to(model2.device)

outputs = model2.generate(
    **inputs,
    max_new_tokens=1024 if ENABLE_THINKING else 384,
    temperature=GEN_TEMPERATURE,
    top_p=GEN_TOP_P,
    top_k=GEN_TOP_K,
    do_sample=True,
)

response = tokenizer2.decode(outputs[0][inputs["input_ids"].shape[-1]:], skip_special_tokens=True)

print(f"\nADAPTER RELOAD TEST (persona: {test_key}):")
print(f"  Q: {TEST_PROMPT}")
print(f"  A: {response[:500]}")
print(f"\nAdapter loads cleanly from disk and carries only language-model tensors.")
print(f"  Sentinel: step {reloaded_sentinel['global_step']}, "
      f"loss {reloaded_sentinel['final_loss']:.4f}, "
      f"{reloaded_sentinel['adapted_modules']} adapted modules")

# List adapter files
print(f"\nAdapter contents:")
for p in sorted(Path(LORA_OUTPUT_DIR).iterdir()):
    size_mb = p.stat().st_size / 1024 / 1024
    print(f"  {p.name:40s} {size_mb:>8.1f} MB")

del model2, tokenizer2, inputs, outputs
gc.collect()
torch.cuda.empty_cache()

Cleared training model from memory
  Loading adapter from: /workspace/training/biblical/output/biblical_qwen3_5_4b_unsloth_4bit/lora_adapters
==((====))==  Unsloth 2026.8.22: Fast Qwen3_5 patching. Transformers: 5.5.0.
   \\   /|    NVIDIA GB10. Num GPUs = 1. Max memory: 121.689 GB. Platform: Linux.
O^O/ \_/ \    Torch: 2.10.0a0+b558c986e8.nv25.11. CUDA: 12.1. CUDA Toolkit: 13.0. Triton: 3.4.0
\        /    Bfloat16 = TRUE. FA [Xformers = 0.0.33+aa7bc36.d20260302. FA2 = True]
 "-____-"     Free license: http://github.com/unslothai/unsloth
Unsloth: Fast downloading is enabled - ignore downloading bars which are red colored!


Loading weights:   0%|          | 0/723 [00:00<?, ?it/s]


Saved adapter tensor families: {'language': 256}

ADAPTER RELOAD TEST (persona: daniel):
  Q: I am struggling with forgiveness. What does Scripture teach about forgiving others?
  A: In the third year of King Jehoiakim’s reign, I saw a ram with two horns, one higher than the other, and I understood that the horn which was higher was the Medes and Persians, who would rise up against the kingdom of Babylon. Yet even as I beheld the rise and fall of empires in vision, I was taught that no kingdom endures by its own strength.

Now you ask of forgiveness—of how one who bears deep wounds may release the grip of resentment. I have stood before kings who demanded justice, not mercy;

Adapter loads cleanly from disk and carries only language-model tensors.
  Sentinel: step 544, loss 1.6775, 128 adapted modules

Adapter contents:
  README.md                                     0.0 MB
  adapter_config.json                           0.0 MB
  adapter_model.safetensors                   162.0 MB
  

## 13. Export Merged Model to GGUF — optional

Merge the LoRA adapter into the base and export to GGUF so it runs anywhere llama.cpp / Ollama /
LM Studio / an iOS GGUF runner is supported. At 4B this is the one model in this family that is
genuinely comfortable on a phone.

- `q4_k_m` is the recommended quant for phones.
- `q5_k_m` and `q8_0` are also produced for higher-quality desktop use.
- Requires Unsloth's bundled `llama.cpp` build (downloaded automatically on first call).

**Not needed for the vLLM path** — the LoRA adapter saved in Cell 10 is the shipping artifact.

**Memory:** this reloads the base at `load_in_4bit=False`, i.e. ~8.7 GB of bf16 weights, then
merges. Comfortable on this machine, unlike the 27B equivalent.

In [13]:
# Compatibility shim for PEFT + GPTQModel during GGUF export.
# PEFT imports GPTQModel's older AWQ class name even when this adapter is not AWQ.
try:
    import gptqmodel.nn_modules.qlinear.gemm_awq as _gemm_awq
    if not hasattr(_gemm_awq, "AwqGEMMQuantLinear") and hasattr(_gemm_awq, "AwqGEMMLinear"):
        _gemm_awq.AwqGEMMQuantLinear = _gemm_awq.AwqGEMMLinear
        print("Patched GPTQModel AWQ class alias for PEFT compatibility.")
except Exception as exc:
    print(f"GPTQModel AWQ compatibility shim skipped: {exc}")

WARN  Python GIL is enabled: Multi-gpu quant acceleration for MoE models is sub-optimal and multi-core accelerated cpu packing is also disabled. We recommend Python >= 3.13.3t with Pytorch > 2.8 for mult-gpu quantization and multi-cpu packing with env `PYTHON_GIL=0`.


INFO  ENV: Auto setting CUDA_DEVICE_ORDER=PCI_BUS_ID for correctness.          


fatal: detected dubious ownership in repository at '/workspace/training/biblical'
To add an exception for this directory, call:

	git config --global --add safe.directory /workspace/training/biblical


INFO  

┌─────────────┐    ┌────────────────────────┐    ┌────────────┐    ┌─────────┐
│ GPT-QModel  │ -> │ ▓▓▓▓▓▓▓▓▓▓▓▓ 16bit     │ -> │ ▒▒▒▒ 8bit  │ -> │ ░░ 4bit │
└─────────────┘    └────────────────────────┘    └────────────┘    └─────────┘
GPT-QModel   : 7.0.0
Transformers : 5.5.0
Torch        : 2.10.0a0+b558c986e8.nv25.11
Triton       : 3.4.0+gitc5d671f9


Patched GPTQModel AWQ class alias for PEFT compatibility.


In [14]:
# Reload the adapter fresh so we export from a clean state
import gc, torch
from pathlib import Path
from unsloth import FastLanguageModel

try:
    del model2, tokenizer2
except NameError:
    pass
gc.collect()
torch.cuda.empty_cache()

GGUF_OUTPUT_DIR = f"{OUTPUT_BASE_DIR}/gguf"
Path(GGUF_OUTPUT_DIR).mkdir(parents=True, exist_ok=True)

# Load base + adapter; Unsloth's GGUF exporter merges before conversion
model_gguf, tokenizer_gguf = FastLanguageModel.from_pretrained(
    model_name=LORA_OUTPUT_DIR,
    max_seq_length=MAX_SEQ_LENGTH,
    load_in_4bit=False,   # full-precision merge for accurate GGUF quantization
)

# ---- Force the chat template's end-of-turn token as the GGUF EOS ----
# REQUIRED for this checkpoint, not defensive boilerplate. Verified in its config.json and
# tokenizer_config.json:
#     config.text_config.eos_token_id = 248044  ->  <|endoftext|>
#     tokenizer eos_token / template close       ->  <|im_end|> (248046)
# llama.cpp's converter reads eos_token_id from the NESTED text_config on
# Conditional-Generation models, so without this override it bakes in <|endoftext|>, the model
# never stops on <|im_end|>, and it runs on into the next turn header. iOS GGUF runners then
# strip the leading <|im_start|> special token and render bare role text ("user\n...") plus a
# hallucinated follow-up question.
_render = tokenizer_gguf.apply_chat_template(
    [{"role": "user", "content": "x"}, {"role": "assistant", "content": "y"}],
    tokenize=False,
)
_eot_candidates = ["<|im_end|>", "<end_of_turn>", "<|eot_id|>"]
eot_token = next((c for c in _eot_candidates if c in _render), None)
if eot_token is None:
    raise RuntimeError(f"Could not detect end-of-turn marker in chat template. Rendered: {_render!r}")

# The Processor wraps the tokenizer; unwrap so tokenizer methods are reachable.
_tok = getattr(tokenizer_gguf, "tokenizer", tokenizer_gguf)
eot_id = _tok.convert_tokens_to_ids(eot_token)
if eot_id is None or eot_id == _tok.unk_token_id:
    raise RuntimeError(f"{eot_token!r} not in tokenizer vocab (got id={eot_id})")

_tok.eos_token = eot_token
model_gguf.config.eos_token_id = eot_id
# The one that actually matters: the nested text_config the converter reads from.
if getattr(model_gguf.config, "text_config", None) is not None:
    model_gguf.config.text_config.eos_token_id = eot_id
# generation_config.eos_token_id may be a list; collapse to the single chat-template EOS so
# runners that honor only a scalar pick the right one.
if getattr(model_gguf, "generation_config", None) is not None:
    model_gguf.generation_config.eos_token_id = eot_id

print(f"Forced GGUF EOS to {eot_token!r} (id={eot_id}); "
      f"text_config.eos_token_id was {248044} (<|endoftext|>)")

# Pass ALL quant methods in one call so the LoRA->FP16 merge happens ONCE and llama.cpp quantizes
# from that single merged file. A loop would re-merge and re-write the FP16 GGUF per quant level.
QUANT_METHODS = ["q4_k_m", "q5_k_m", "q8_0"]

print(f"Exporting GGUF (single merge -> {len(QUANT_METHODS)} quants)...")
model_gguf.save_pretrained_gguf(
    GGUF_OUTPUT_DIR,
    tokenizer_gguf,
    quantization_method=QUANT_METHODS,
)

print(f"\nGGUF export complete: {GGUF_OUTPUT_DIR}")
for f in sorted(Path(GGUF_OUTPUT_DIR).glob("*.gguf")):
    size_mb = f.stat().st_size / 1024 / 1024
    print(f"  {f.name:60s} {size_mb:>8.1f} MB")

print("\nMobile usage (iPhone):")
print("  1. Transfer the q4_k_m .gguf file to the phone (AirDrop / Files app).")
print("  2. Open in an iOS GGUF runner (LLMFarm, PocketPal, Private LLM, etc.).")
print("  3. Use the Qwen ChatML template; set context length <= MAX_SEQ_LENGTH.")

del model_gguf, tokenizer_gguf
gc.collect()
torch.cuda.empty_cache()

==((====))==  Unsloth 2026.8.22: Fast Qwen3_5 patching. Transformers: 5.5.0.
   \\   /|    NVIDIA GB10. Num GPUs = 1. Max memory: 121.689 GB. Platform: Linux.
O^O/ \_/ \    Torch: 2.10.0a0+b558c986e8.nv25.11. CUDA: 12.1. CUDA Toolkit: 13.0. Triton: 3.4.0
\        /    Bfloat16 = TRUE. FA [Xformers = 0.0.33+aa7bc36.d20260302. FA2 = True]
 "-____-"     Free license: http://github.com/unslothai/unsloth
Unsloth: Fast downloading is enabled - ignore downloading bars which are red colored!
Unsloth: QLoRA and full finetuning all not selected. Switching to 16bit LoRA.


Loading weights:   0%|          | 0/723 [00:00<?, ?it/s]

Forced GGUF EOS to '<|im_end|>' (id=248046); text_config.eos_token_id was 248044 (<|endoftext|>)
Exporting GGUF (single merge -> 3 quants)...
Unsloth: Merging model weights to 16-bit format...


Unsloth: Restored added_tokens_decoder metadata in /workspace/training/biblical/output/biblical_qwen3_5_4b_unsloth_4bit/gguf/tokenizer_config.json.
Unsloth: Restored added_tokens_decoder metadata in /workspace/training/biblical/output/biblical_qwen3_5_4b_unsloth_4bit/gguf/tokenizer_config.json.


Found HuggingFace hub cache directory: /root/.cache/huggingface


...


Unsloth: Preparing safetensor model files: 100%|██████████| 2/2 [00:00<00:00, 76260.07it/s]

Unsloth: Merging weights into 16bit: 100%|██████████| 2/2 [00:47<00:00, 23.82s/it]


Unsloth: Merge process complete. Saved to `/workspace/training/biblical/output/biblical_qwen3_5_4b_unsloth_4bit/gguf`
Unsloth: Converting to GGUF format...
==((====))==  Unsloth: Conversion from HF to GGUF information
   \\   /|    [0] Installing llama.cpp might take 3 minutes.
O^O/ \_/ \    [1] Converting HF to GGUF bf16 might take 3 minutes.
\        /    [2] Converting GGUF bf16 to ['q4_k_m', 'q5_k_m', 'q8_0'] might take 10 minutes each.
 "-____-"     In total, you will have to wait at least 16 minutes.

Unsloth: llama.cpp found in the system. Skipping installation.
Unsloth: Preparing converter script...


[unsloth_zoo.llama_cpp|WARNING]Unsloth: No supported architectures (TEXT or VISION) could be determined from the original script.
[unsloth_zoo.llama_cpp|WARNING]Unsloth: Metadata branding patch target 'self.metadata = gguf.Metadata.load(...)' not found.
[unsloth_zoo.llama_cpp|WARNING]Unsloth: Qwen2MoE num_experts patch target not found.


Unsloth: [1] Converting model into bf16 GGUF format.
This might take 3 minutes...
Unsloth: Qwen3_5ForConditionalGeneration is not supported for MMPROJ conversion. Converting as text-only model.


RuntimeError: Unsloth: GGUF conversion failed: RuntimeError: Unsloth: Failed to convert model to GGUF with command `/usr/bin/python /root/.unsloth/llama.cpp/unsloth_convert_hf_to_gguf.py --outfile Qwen3.5-4B.BF16.gguf --outtype bf16 --split-max-size 50G /workspace/training/biblical/output/biblical_qwen3_5_4b_unsloth_4bit/gguf`: Command '['/usr/bin/python', '/root/.unsloth/llama.cpp/unsloth_convert_hf_to_gguf.py', '--outfile', 'Qwen3.5-4B.BF16.gguf', '--outtype', 'bf16', '--split-max-size', '50G', '/workspace/training/biblical/output/biblical_qwen3_5_4b_unsloth_4bit/gguf']' returned non-zero exit status 1.
--- converter stderr ---
Traceback (most recent call last):
  File "/root/.unsloth/llama.cpp/unsloth_convert_hf_to_gguf.py", line 18, in <module>
    from conversion import (
ModuleNotFoundError: No module named 'conversion'